# HW9 on Data Science course of Sharif University of Technology
## Created by: Mohammad Mahdi Hossein Beiky     SI: 400100995
## GitHub URL: https://github.com/Mmhb1382/Data_Science_HW9.git
---

### Preprocessing steps

In [1]:
# Import libraries for data handling and preprocessing
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Load the Ames Housing dataset
df = pd.read_csv('AmesHousing.csv')

# === Feature Engineering ===
# Age of the house at time of sale
df['HouseAge'] = df['Yr Sold'] - df['Year Built']
# Years since last remodel
df['RemodelAge'] = df['Yr Sold'] - df['Year Remod/Add']
# Total finished square feet (including basement)
df['TotalSF'] = df['Total Bsmt SF'].fillna(0) + df['1st Flr SF'] + df['2nd Flr SF']

# === Prepare data for modeling ===
# Drop identifier columns that won't help the model
df_model = df.drop(['Order', 'PID'], axis=1)
# Separate target variable
y = df_model['SalePrice']

# Identify which columns are numeric vs. categorical
numeric_features = (
    df_model.select_dtypes(include=['int64', 'float64'])
            .columns
            .drop('SalePrice')
            .tolist()
)
categorical_features = df_model.select_dtypes(include=['object']).columns.tolist()

# === Build preprocessing pipelines ===
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),   # Fill missing numeric values
    ('scale', StandardScaler())                     # Standardize to zero mean & unit variance
])

categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))        # One-hot encode categories
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer([
    ('nums', numeric_pipeline, numeric_features),
    ('cats', categorical_pipeline, categorical_features)
])

# === Apply preprocessing ===
X_processed = preprocessor.fit_transform(df_model)

# Map back to a DataFrame with human-readable column names
feature_names = preprocessor.get_feature_names_out()
X_df = pd.DataFrame(X_processed, columns=feature_names)

# Display the first few rows of the prepared feature set
X_df.head()


,nums__MS SubClass,nums__Lot Frontage,nums__Lot Area,nums__Overall Qual,nums__Overall Cond,nums__Year Built,nums__Year Remod/Add,nums__Mas Vnr Area,nums__BsmtFin SF 1,nums__BsmtFin SF 2,...,cats__Sale Type_New,cats__Sale Type_Oth,cats__Sale Type_VWD,cats__Sale Type_WD,cats__Sale Condition_Abnorml,cats__Sale Condition_AdjLand,cats__Sale Condition_Alloca,cats__Sale Condition_Family,cats__Sale Condition_Normal,cats__Sale Condition_Partial
0,-0.877005,3.375742,2.744381,-0.067254,-0.506718,-0.375537,-1.163488,0.061046,0.431223,-0.293918,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.877005,0.514952,0.187097,-0.776079,0.393091,-0.342468,-1.115542,-0.566039,0.055760,0.557582,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.877005,0.561850,0.522814,-0.067254,0.393091,-0.441674,-1.259380,0.038650,1.054800,-0.293918,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.877005,1.124628,0.128458,0.641571,-0.506718,-0.110988,-0.779919,-0.566039,1.366588,-0.293918,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.061285,0.233563,0.467348,-0.776079,-0.506718,0.848000,0.658466,-0.566039,0.764969,-0.293918,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


### First Task for binary classification

In [2]:
# Import necessary model and evaluation tools
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import f1_score, r2_score

# === Prepare a binary classification target ===
# Label houses above the median price as 1 (expensive), others as 0
y_class = (y > y.median()).astype(int)

# === Split data into training and test sets ===
# Use 80% for training and 20% for testing
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_df, y_class, test_size=0.2, random_state=42
)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_df, y, test_size=0.2, random_state=42
)

# === Train a multilayer perceptron for classification ===
# Here: one hidden layer of 100 neurons, up to 500 iterations
clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
clf.fit(X_train_cls, y_train_cls)

# === Evaluate classification performance ===
y_pred_cls = clf.predict(X_test_cls)
cls_f1 = f1_score(y_test_cls, y_pred_cls)
print(f"Classification F1-score: {cls_f1:.3f}")


Classification F1-score: 0.941


### First task for Regression

In [3]:
# === Train a multilayer perceptron for regression ===
reg_tuned = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),    # three hidden layers of decreasing size
    solver='adam',                        # stochastic optimizer
    alpha=1e-3,                           # L2 regularization strength
    learning_rate='adaptive',             # reduce lr when plateaued
    learning_rate_init=1e-3,              # starting learning rate
    max_iter=2000,                        # allow more epochs for convergence
    early_stopping=True,                  # stop if no improvement on validation
    validation_fraction=0.1,              # 10% of train set for validation
    n_iter_no_change=30,                  # patience before early stop
    random_state=42
)

# Fit and evaluate
reg_tuned.fit(X_train_reg, y_train_reg)
y_pred_tuned = reg_tuned.predict(X_test_reg)
r2_tuned = r2_score(y_test_reg, y_pred_tuned)
print(f"Tuned Regression R² score: {r2_tuned:.3f}")


Tuned Regression R² score: 0.897


### Second task for both binary classification and Regression

In [4]:
# ===== Task 2: 4-Layer Feedforward Network with Keras =====

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score, r2_score

# Prepare classification labels: 1 if SalePrice above median, else 0
y_class = (y > y.median()).astype(int)

# Split into train/test sets (80/20) for both tasks, using NumPy arrays to avoid feature‐name issues
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_df.values, y_class.values, test_size=0.2, random_state=42
)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_df.values, y.values, test_size=0.2, random_state=42
)

# --- Classification Model ---
model_clf = Sequential([
    Input(shape=(X_train_cls.shape[1],)),     # define inputs here
    Dense(128, activation='relu'),            # hidden layer 1
    Dense(64,  activation='relu'),            # hidden layer 2
    Dense(32,  activation='relu'),            # hidden layer 3
    Dense(16,  activation='relu'),            # hidden layer 4
    Dense(1,   activation='sigmoid')          # output layer
])
model_clf.compile(optimizer=Adam(1e-3), loss='binary_crossentropy')
model_clf.fit(X_train_cls, y_train_cls, validation_split=0.1, epochs=50, batch_size=32, verbose=1)
y_pred_cls = (model_clf.predict(X_test_cls).ravel() >= 0.5).astype(int)
print(f"Keras Classification F1-score: {f1_score(y_test_cls, y_pred_cls):.3f}")

# --- Regression Model ---
model_reg = Sequential([
    Input(shape=(X_train_reg.shape[1],)),     # explicit Input here too
    Dense(128, activation='relu'),
    Dense(64,  activation='relu'),
    Dense(32,  activation='relu'),
    Dense(16,  activation='relu'),
    Dense(1,   activation='linear')           # linear output
])
model_reg.compile(optimizer=Adam(1e-3), loss='mse')
model_reg.fit(X_train_reg, y_train_reg, validation_split=0.1, epochs=100, batch_size=32, verbose=1)
y_pred_reg = model_reg.predict(X_test_reg).ravel()
print(f"Keras Regression R² score: {r2_score(y_test_reg, y_pred_reg):.3f}")


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Epoch 1/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4470 - val_loss: 0.2181
Epoch 2/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1610 - val_loss: 0.2199
Epoch 3/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1374 - val_loss: 0.2398
Epoch 4/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1098 - val_loss: 0.2361
Epoch 5/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0861 - val_loss: 0.2223
Epoch 6/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0907 - val_loss: 0.2598
Epoch 7/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0604 - val_loss: 0.2887
Epoch 8/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0459 - val_loss: 0.3286
Epoch 9/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0346 - val_loss: 0.3392
Epoch 10/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0271 - val_loss: 0.3778
Epoch 11/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0207 - val_loss: 0.4529
Epoch 12/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0238 - val_lo

#### Keras Classification F1-score: 0.941
#### Keras Regression R² score: 0.906

### Third task

In [5]:
# ===== Task 3: 4-Layer Feedforward Network with PyTorch =====

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, r2_score

# --- 1) Turn our existing NumPy train/test splits into PyTorch tensors ---
# (X_train_cls, X_test_cls, etc. are already NumPy arrays from Task 1/2)
X_train_cls_t = torch.tensor(X_train_cls, dtype=torch.float32)
y_train_cls_t = torch.tensor(y_train_cls, dtype=torch.float32)
X_test_cls_t  = torch.tensor(X_test_cls,  dtype=torch.float32)
y_test_cls_t  = torch.tensor(y_test_cls,  dtype=torch.float32)

X_train_reg_t = torch.tensor(X_train_reg, dtype=torch.float32)
y_train_reg_t = torch.tensor(y_train_reg, dtype=torch.float32).unsqueeze(1)
X_test_reg_t  = torch.tensor(X_test_reg,  dtype=torch.float32)
y_test_reg_t  = torch.tensor(y_test_reg,  dtype=torch.float32).unsqueeze(1)

# --- 2) Create DataLoaders for batching ---
batch_size = 32
train_cls_loader = DataLoader(TensorDataset(X_train_cls_t, y_train_cls_t),
                              batch_size=batch_size, shuffle=True)
train_reg_loader = DataLoader(TensorDataset(X_train_reg_t, y_train_reg_t),
                              batch_size=batch_size, shuffle=True)

# --- 3) Define a reusable 4-layer MLP class ---
class FeedforwardNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128,       64),  nn.ReLU(),
            nn.Linear(64,        32),  nn.ReLU(),
            nn.Linear(32,        16),  nn.ReLU(),
            nn.Linear(16,  output_dim)
        )
    def forward(self, x):
        return self.net(x)

# === Classification: BCEWithLogitsLoss combines sigmoid + binary CE ===
model_cls   = FeedforwardNet(X_train_cls_t.shape[1], 1)
criterion_c = nn.BCEWithLogitsLoss()
optimizer_c = optim.Adam(model_cls.parameters(), lr=1e-3)

for epoch in range(50):                # train for 50 epochs
    model_cls.train()
    for Xb, yb in train_cls_loader:
        optimizer_c.zero_grad()
        logits = model_cls(Xb).squeeze()
        loss   = criterion_c(logits, yb)
        loss.backward()
        optimizer_c.step()

# Evaluate on test set
model_cls.eval()
with torch.no_grad():
    logits_test = model_cls(X_test_cls_t).squeeze()
    probs       = torch.sigmoid(logits_test).numpy()
    preds       = (probs >= 0.5).astype(int)
    cls_f1      = f1_score(y_test_cls, preds)
print(f"PyTorch Classification F1-score: {cls_f1:.3f}")

# === Regression: standard MSE loss ===
model_reg   = FeedforwardNet(X_train_reg_t.shape[1], 1)
criterion_r = nn.MSELoss()
optimizer_r = optim.Adam(model_reg.parameters(), lr=1e-3)

for epoch in range(50):                # train for 50 epochs
    model_reg.train()
    for Xb, yb in train_reg_loader:
        optimizer_r.zero_grad()
        out  = model_reg(Xb)
        loss = criterion_r(out, yb)
        loss.backward()
        optimizer_r.step()

# Evaluate on test set
model_reg.eval()
with torch.no_grad():
    preds_r = model_reg(X_test_reg_t).squeeze().numpy()
    reg_r2  = r2_score(y_test_reg, preds_r)
print(f"PyTorch Regression R² score: {reg_r2:.3f}")


PyTorch Classification F1-score: 0.928
PyTorch Regression R² score: 0.896


### Fourth Task

In [6]:
# ===== Task 4: 4‑Layer Non‑Sequential Feedforward Network with Keras (Functional API) =====

import numpy as np
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score, r2_score

# --- 1) Ensure our splits are NumPy arrays ---
# If they’re still DataFrames/Series, grab .values; otherwise assume they’re already arrays
X_train_cls_np = X_train_cls.values if hasattr(X_train_cls, "values") else X_train_cls
X_test_cls_np  = X_test_cls.values  if hasattr(X_test_cls,  "values") else X_test_cls
y_train_cls_np = y_train_cls.values if hasattr(y_train_cls, "values") else y_train_cls
y_test_cls_np  = y_test_cls.values  if hasattr(y_test_cls,  "values") else y_test_cls

X_train_reg_np = X_train_reg.values if hasattr(X_train_reg, "values") else X_train_reg
X_test_reg_np  = X_test_reg.values  if hasattr(X_test_reg,  "values") else X_test_reg
y_train_reg_np = y_train_reg.values if hasattr(y_train_reg, "values") else y_train_reg
y_test_reg_np  = y_test_reg.values  if hasattr(y_test_reg,  "values") else y_test_reg

# --- 2) Classification model (Functional API) ---
# Define input layer matching our number of features
inputs = Input(shape=(X_train_cls_np.shape[1],), name="clf_input")
# Stack four Dense+ReLU layers
x = Dense(128, activation="relu", name="clf_dense1")(inputs)
x = Dense(64,  activation="relu", name="clf_dense2")(x)
x = Dense(32,  activation="relu", name="clf_dense3")(x)
x = Dense(16,  activation="relu", name="clf_dense4")(x)
# Final sigmoid output for binary classification
outputs = Dense(1, activation="sigmoid", name="clf_output")(x)

model_clf = Model(inputs, outputs, name="functional_classifier")
model_clf.compile(optimizer=Adam(1e-3), loss="binary_crossentropy")

# Train, reserving 10% of training set for validation
model_clf.fit(
    X_train_cls_np, y_train_cls_np,
    validation_split=0.1, epochs=50, batch_size=32, verbose=1
)

# Evaluate on test set
y_pred_cls = (model_clf.predict(X_test_cls_np).ravel() >= 0.5).astype(int)
print(f"Functional Keras Classification F1-score: {f1_score(y_test_cls_np, y_pred_cls):.3f}")

# --- 3) Regression model (Functional API) ---
inputs = Input(shape=(X_train_reg_np.shape[1],), name="reg_input")
x = Dense(128, activation="relu", name="reg_dense1")(inputs)
x = Dense(64,  activation="relu", name="reg_dense2")(x)
x = Dense(32,  activation="relu", name="reg_dense3")(x)
x = Dense(16,  activation="relu", name="reg_dense4")(x)
# Linear output for continuous regression
outputs = Dense(1, activation="linear", name="reg_output")(x)

model_reg = Model(inputs, outputs, name="functional_regressor")
model_reg.compile(optimizer=Adam(1e-3), loss="mse")

model_reg.fit(
    X_train_reg_np, y_train_reg_np,
    validation_split=0.1, epochs=100, batch_size=32, verbose=1
)

y_pred_reg = model_reg.predict(X_test_reg_np).ravel()
print(f"Functional Keras Regression R² score: {r2_score(y_test_reg_np, y_pred_reg):.3f}")


Epoch 1/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3931 - val_loss: 0.2190
Epoch 2/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1705 - val_loss: 0.2229
Epoch 3/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1359 - val_loss: 0.2206
Epoch 4/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1073 - val_loss: 0.2414
Epoch 5/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1004 - val_loss: 0.2496
Epoch 6/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0663 - val_loss: 0.3045
Epoch 7/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0647 - val_loss: 0.2956
Epoch 8/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0404 - val_loss: 0.3540
Epoch 9/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0264 - val_loss: 0.3448
Epoch 10/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0242 - val_loss: 0.4404
Epoch 11/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0240 - val_loss: 0.4051
Epoch 12/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0177 - val_lo

#### Functional Keras Classification F1-score: 0.938
#### Functional Keras Regression R² score: 0.900

### Fifth task

## Why Neural Networks Are So Powerful

1. **Universal Function Approximation**
   Neural networks—given enough hidden units—can approximate any continuous function to arbitrary precision. This means they can model extremely complex, non-linear relationships in data that simple linear or tree-based models might miss.

2. **Automatic Feature Learning**
   Instead of relying on hand-crafted features, deep networks learn hierarchical representations directly from raw inputs. Early layers capture low-level patterns (edges, textures), while deeper layers combine these into higher‐level concepts (rooms, layouts, price drivers).

3. **Flexibility Across Domains**
   The same basic building blocks (Dense, Convolutional, Recurrent layers) can be composed to solve vision, language, time-series, and structured-tabular problems—often with minimal changes to the overall framework.

4. **End-to-End Training**
   Everything from raw inputs to final outputs is optimized jointly via gradient descent. This holistic approach often yields better performance than “piecemeal” pipelines where different components are tuned in isolation.

---

## The Hard Part: Designing & Tuning Neural Networks

1. **Architecture Selection**
   How many layers? How many neurons per layer? Dense vs. convolution vs. recurrent? Every choice changes both learning capacity and computational cost. There’s no one-size-fits-all recipe.

2. **Hyperparameter Tuning**
   Learning rate schedules, batch size, regularization strength (L₂, dropout), optimizer choice—all can drastically alter convergence speed and generalization. Grid or random search helps, but it’s time-consuming.

3. **Overfitting vs. Underfitting**
   Too much capacity leads to memorization; too little, and the network can’t capture important patterns. Balancing model size, early stopping, and regularization is part art, part science.

4. **Vanishing/Exploding Gradients**
   Deep stacks of layers can suffer from gradients that shrink or blow up, making training unstable. Architectural tricks (residual connections, batch norm) and careful initialization help—but add complexity.

5. **Data & Compute Requirements**
   Large networks often need massive datasets to generalize well, as well as GPUs or TPUs to train in reasonable time. For many tabular problems, simpler methods can compete if data is scarce.

---

*With these trade-offs in mind, the real “magic” of neural networks comes from combining their representational power with robust validation, sensible regularization, and automated search (e.g. AutoML, Bayesian optimization) to find the right configuration for your problem.*
